# Лабораторная 3 Численные методы

Дан интеграл вида: $$\int^b_a f(x) dx$$ где $a = 0.35$, $b = 1.35$, $f(x)=0.35 e^x + 0.65 \cos x$.

Для вычисления интеграла с точностью $\varepsilon = 10^{-5}$ необходимо:

1. Пользуясь выражением для погрешности интегрирования, определить шаг $h$ в составной квадратурной формуле, которая обеспечит требуемую точность результата.
Рассмотреть квадратурная формулу Симпсона.
2. Для СКФ из п.1 определить величину $h$ шага разбиения исходного отрезка интегрирования, достаточного для достижения точности ε, по правилу Рунге.
3. Применить квадратурную формулу НАСТ Гаусса при указанном значении n. Оценить погрешность интегрирования через формулу остаточного члена $R_n(f)$. $n=1$.
4. Провести сравнительный анализ полученных в п.п.1-3 результатов.

### Аналитический вывод решения

$$\int^{1.35}_{0.35} 0.35 e^x + 0.65 \cos x = \left. (0.35 e^x + 0.65  \sin x) \right|_{0.35}^{1.35} \approx 1.264761901477586 $$

In [2]:
import numpy as np

def func(x):
    return 0.35 * np.exp(x) + 0.65 * np.cos(x)

def simpson(a, b, n):
    h = (b - a) / n
    nodes = np.linspace(a, b, n+1)
    y = func(nodes)
    return h/3 * (y[0] + y[-1] + 4*np.sum(y[1:-1:2]) + 2*np.sum(y[2:-2:2]))

a, b, eps = 0.35, 1.35, 10**(-5)

ANALYTICAL = 0.35 * (np.exp(1.35) - np.exp(0.35)) + 0.65 * (np.sin(1.35) - np.sin(0.35))
ANALYTICAL

np.float64(1.264761901477586)

### 1. Определение шага интегрирования при помощи выражения для погрешности

Для оценки количества разбиений отрезка воспользуемся формулой: $$N_{\text{сим}} \ge  \sqrt[4]{\frac{(b-a)^5 M_4}{2880 \varepsilon}},\ \ M_4= \max |f^{(4)} (x)|$$

В контексте данной задачи: $$\begin{gathered} M_4 = \max |f^{(4)}(\xi) = 0.35 e^x + 0.65 \cos x|=1.49245, \\ N_{\text{сим}} \ge \sqrt[4]{\frac{1.49245}{2880 \cdot 10^{-5}}} \end{gathered}$$

In [3]:
M = func(b)
R = (M / (180 * eps))
n_apr = R ** (1/4)
n_apr

np.float64(5.366078975029479)

Таким образом, $N_{\text{сим}} \ge 5.366$, значит, число разбиений, необходимое для достижения заданной точности, равняется $6$. Вычислим приближённое значение интеграла по квадратурной формуле Симпсона:

In [ ]:
N_1 = 6
I1 = simpson(a, b, N_1)
err1 = np.abs(ANALYTICAL - I1)
print(I1, err1)

1.2647636158105686 1.7143329826829756e-06


### 2. Определение шага по правилу Рунге

In [20]:
def runge_rule(formula, func, a, b):
    N_prev= 2
    I_prev = formula(a, b, N_prev)
    while True:
        N_next = N_prev * 2
        I_next = formula(a, b, N_next)
        R_runge = (I_next - I_prev) / (1 - N_prev/N_next)
        if np.abs(R_runge) < eps:
            break
        N_prev, I_prev = N_next, I_next
    I = I_prev + (I_next - I_prev) / (1 - (N_prev / N_next))
    return I, N_prev
    
I_2, N_2 = runge_rule(simpson, func, a, b)
err2 = np.abs(I_2 - ANALYTICAL)
I_2, N_2, err2

(np.float64(1.2647604015401788), 8, np.float64(1.4999374071678062e-06))